# Gradient Descent vs. Adam Optimizer

This notebook explores two optimization algorithms used in training machine learning models:
1.  **Gradient Descent**: The fundamental optimization algorithm.
2.  **Adam (Adaptive Moment Estimation)**: A popular, advanced optimizer that combines ideas from Momentum and RMSProp.

We will implement both from scratch to understand their mechanics and compare their paths to the minimum.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define a simple convex function to minimize: f(x) = x^2
def f(x):
    return x**2

# Define the gradient (derivative) of the function: f'(x) = 2x
def gradient(x):
    return 2*x

## 1. Gradient Descent

**Concept:** Gradient Descent takes steps proportional to the negative of the gradient (slope) of the function at the current point. It's like walking down a hill by always taking a step in the steepest downhill direction.

**Update Rule:**
$$\theta_{t+1} = \theta_t - \eta \cdot \nabla f(\theta_t)$$

Where:
- $\theta_t$: Current parameter value (position).
- $\eta$ (eta): Learning rate (step size).
- $\nabla f(\theta_t)$: Gradient at the current position.

In [ ]:
def gradient_descent(start_x, learning_rate, num_iterations):
    x = start_x
    history = [x]
    
    for _ in range(num_iterations):
        grad = gradient(x)
        x = x - learning_rate * grad
        history.append(x)
        
    return history

## 2. Adam Optimizer

**Concept:** Adam (Adaptive Moment Estimation) computes adaptive learning rates for each parameter. It stores both an exponentially decaying average of past gradients (**Momentum**) and an exponentially decaying average of past squared gradients (**RMSProp**).

**Key Components:**
1.  **First Moment ($m_t$)**: Estimate of the mean of the gradients (Momentum).
2.  **Second Moment ($v_t$)**: Estimate of the uncentered variance of the gradients (RMSProp).
3.  **Bias Correction**: Corrects $m_t$ and $v_t$ towards zero during initial steps.

**Formulas:**

1.  **Update biased first moment estimate:**
    $$m_t = \beta_1 \cdot m_{t-1} + (1 - \beta_1) \cdot g_t$$

2.  **Update biased second raw moment estimate:**
    $$v_t = \beta_2 \cdot v_{t-1} + (1 - \beta_2) \cdot g_t^2$$

3.  **Compute bias-corrected first moment estimate:**
    $$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$$

4.  **Compute bias-corrected second raw moment estimate:**
    $$\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

5.  **Update parameters:**
    $$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \cdot \hat{m}_t$$

Where:
- $g_t$: Gradient at time step $t$.
- $\beta_1$: Exponential decay rate for the first moment estimates (typically 0.9).
- $\beta_2$: Exponential decay rate for the second moment estimates (typically 0.999).
- $\epsilon$: Small constant to prevent division by zero (typically $10^{-8}$).

In [ ]:
def adam_optimizer(start_x, learning_rate, num_iterations, beta1=0.9, beta2=0.999, epsilon=1e-8):
    x = start_x
    history = [x]
    
    # Initialize moments
    m = 0
    v = 0
    
    for t in range(1, num_iterations + 1):
        g = gradient(x)
        
        # Update biased first moment estimate
        m = beta1 * m + (1 - beta1) * g
        
        # Update biased second raw moment estimate
        v = beta2 * v + (1 - beta2) * (g**2)
        
        # Compute bias-corrected first moment estimate
        m_hat = m / (1 - beta1**t)
        
        # Compute bias-corrected second raw moment estimate
        v_hat = v / (1 - beta2**t)
        
        # Update parameters
        x = x - learning_rate * m_hat / (np.sqrt(v_hat) + epsilon)
        
        history.append(x)
        
    return history

## 3. Comparison and Visualization

Let's run both optimizers starting from the same point and visualize their paths.

In [ ]:
# Parameters
start_x = -4.0
lr = 0.1
iterations = 50

# Run Optimizers
gd_history = gradient_descent(start_x, lr, iterations)
adam_history = adam_optimizer(start_x, lr, iterations)

# Plotting
x_range = np.linspace(-5, 5, 100)
y_range = f(x_range)

plt.figure(figsize=(12, 6))
plt.plot(x_range, y_range, 'k--', alpha=0.3, label='f(x) = x^2')

# Plot Gradient Descent Path
plt.plot(gd_history, [f(x) for x in gd_history], '-', label='Gradient Descent', color='blue', markersize=2)

# Plot Adam Path
plt.plot(adam_history, [f(x) for x in adam_history], 'o-', label='Adam', color='red', markersize=5)

plt.title(f'Gradient Descent vs Adam (Learning Rate: {lr})')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Observations

- **Gradient Descent**: Takes steps purely based on the slope. With a fixed learning rate, it slows down as the slope gets flatter near the minimum.
- **Adam**: Uses momentum to accelerate in the correct direction and adaptive scaling to normalize the step size. You might notice it approaches the minimum differently, sometimes faster or with more confidence, depending on the hyperparameters.

## TASK

- Try to implement the gradient descent and Adam optimizer from scratch.
- Decrease the learning rate and see the effect.
- Increase number of iterations and see the effect.
- Change the function and see the effect.
- Try to implement `Nesterov optimizer` for comparison.